# Customer Retention & Value Analysis

## Business Problem

A business may generate strong overall sales while still losing valuable customers or failing to retain them.

The objective of this analysis is to understand customer purchasing behavior and identify opportunities to improve customer retention and lifetime value.

### Key Business Question

**Which customers are most valuable, which customers are at risk of being lost, and what actions can the business take to improve customer retention and customer lifetime value?**

### Analysis Objectives

1. Identify the most valuable customers.
2. Understand customer purchasing frequency and behavior.
3. Segment customers based on Recency, Frequency, and Monetary value (RFM).
4. Measure repeat purchasing and customer retention.
5. Identify customers who may be at risk of becoming inactive.
6. Develop business recommendations to improve retention and customer value.

## Dataset

The analysis uses the UCI Online Retail dataset.

The dataset contains transaction-level records from a UK-based online retailer covering December 2010 to December 2011.

Each row represents a transaction line and contains information about:

- Invoice number
- Product
- Quantity
- Invoice date
- Unit price
- Customer
- Country

The dataset contains both customer purchases and non-standard transactions such as cancellations and adjustments. These records will be investigated and handled according to the business objective.

In [1]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\ADMIN\Desktop\p2\Online Retail.xlsx"

orders = pd.read_excel(file_path)

orders.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Initial Data Audit

Before cleaning the dataset, an initial audit was performed to understand its structure, completeness, transaction patterns, and potential data-quality issues.

The audit focuses on:

- Dataset size and structure
- Missing values
- Duplicate records
- Cancellations
- Negative quantities
- Invalid prices
- Customer identifiers
- Transaction and customer counts

In [2]:
print("SHAPE")
print(orders.shape)

print("\nCOLUMNS")
print(orders.columns.tolist())

print("\nDATA TYPES")
print(orders.dtypes)

print("\nMISSING VALUES")
print(orders.isna().sum())

print("\nDUPLICATE ROWS")
print(orders.duplicated().sum())

print("\nDATE RANGE")
print("Start:", orders["InvoiceDate"].min())
print("End:", orders["InvoiceDate"].max())

print("\nUNIQUE CUSTOMERS")
print(orders["CustomerID"].nunique())

print("\nUNIQUE INVOICES")
print(orders["InvoiceNo"].nunique())

print("\nCANCELLED INVOICES")
cancelled = orders["InvoiceNo"].astype(str).str.startswith("C")
print(cancelled.sum())

print("\nNEGATIVE QUANTITIES")
print((orders["Quantity"] < 0).sum())

print("\nZERO / NEGATIVE PRICES")
print((orders["UnitPrice"] <= 0).sum())

SHAPE
(541909, 8)

COLUMNS
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

DATA TYPES
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

MISSING VALUES
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

DUPLICATE ROWS
5268

DATE RANGE
Start: 2010-12-01 08:26:00
End: 2011-12-09 12:50:00

UNIQUE CUSTOMERS
4372

UNIQUE INVOICES
25900

CANCELLED INVOICES
9288

NEGATIVE QUANTITIES
10624

ZERO / NEGATIVE PRICES
2517


In [3]:
audit_summary = {
    "Total rows": len(orders),
    "Unique invoices": orders["InvoiceNo"].nunique(),
    "Unique customers": orders["CustomerID"].nunique(),
    "Missing CustomerID rows": orders["CustomerID"].isna().sum(),
    "Duplicate rows": orders.duplicated().sum(),
    "Cancelled rows": cancelled.sum(),
    "Unique cancelled invoices": orders.loc[cancelled, "InvoiceNo"].nunique(),
    "Negative quantity rows": (orders["Quantity"] < 0).sum(),
    "Zero/negative price rows": (orders["UnitPrice"] <= 0).sum(),
    "Missing descriptions": orders["Description"].isna().sum()
}

audit_summary

{'Total rows': 541909,
 'Unique invoices': 25900,
 'Unique customers': 4372,
 'Missing CustomerID rows': np.int64(135080),
 'Duplicate rows': np.int64(5268),
 'Cancelled rows': np.int64(9288),
 'Unique cancelled invoices': 3836,
 'Negative quantity rows': np.int64(10624),
 'Zero/negative price rows': np.int64(2517),
 'Missing descriptions': np.int64(1454)}

## Investigating Data Quality Issues

The initial audit identified several unusual transaction patterns.

Instead of automatically removing these records, each issue was investigated to determine whether it represented:

- A genuine customer purchase
- A cancellation
- A return or adjustment
- An operational write-off
- An invalid transaction

This prevents potentially meaningful business information from being incorrectly classified as bad data.

In [4]:
# Investigate negative quantities that are NOT cancellations

non_cancel_negative = orders[
    (~orders["InvoiceNo"].astype(str).str.startswith("C")) &
    (orders["Quantity"] < 0)
]

print("Rows:", len(non_cancel_negative))
print("Unique invoices:", non_cancel_negative["InvoiceNo"].nunique())
print("Unique customers:", non_cancel_negative["CustomerID"].nunique())

non_cancel_negative.sort_values("Quantity").head(20)

Rows: 1336
Unique invoices: 1336
Unique customers: 0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
225529,556690,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.0,NaN,United Kingdom
225530,556691,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.0,NaN,United Kingdom
225528,556687,23003,Printing smudges/thrown away,-9058,2011-06-14 10:36:00,0.0,NaN,United Kingdom
115818,546152,72140F,throw away,-5368,2011-03-09 17:25:00,0.0,NaN,United Kingdom
431381,573596,79323W,"Unsaleable, destroyed.",-4830,2011-10-31 15:17:00,0.0,NaN,United Kingdom
341601,566768,16045,NaN,-3667,2011-09-14 17:53:00,0.0,NaN,United Kingdom
323458,565304,16259,NaN,-3167,2011-09-02 12:18:00,0.0,NaN,United Kingdom
263884,560039,20713,wrongly marked. 23343 in box,-3100,2011-07-14 14:27:00,0.0,NaN,United Kingdom
113580,545990,84598,check,-3000,2011-03-08 13:07:00,0.0,NaN,United Kingdom
375429,569466,23270,incorrect stock entry.,-2880,2011-10-04 11:42:00,0.0,NaN,United Kingdom


In [5]:
# Investigate zero and negative prices

invalid_price = orders[orders["UnitPrice"] <= 0]

print("Rows:", len(invalid_price))
print("Unique invoices:", invalid_price["InvoiceNo"].nunique())
print("Unique customers:", invalid_price["CustomerID"].nunique())

invalid_price.sort_values("UnitPrice").head(20)

Rows: 2517
Unique invoices: 2157
Unique customers: 31


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom
536908,581226,23090,missing,-338,2011-12-08 09:56:00,0.00,NaN,United Kingdom
535336,581213,22576,check,-30,2011-12-07 18:38:00,0.00,NaN,United Kingdom
535335,581212,22578,lost,-1050,2011-12-07 18:38:00,0.00,NaN,United Kingdom
535334,581211,22142,check,14,2011-12-07 18:36:00,0.00,NaN,United Kingdom
535333,581210,23395,check,-26,2011-12-07 18:36:00,0.00,NaN,United Kingdom
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.00,NaN,United Kingdom
535331,581208,72801C,check,-10,2011-12-07 18:35:00,0.00,NaN,United Kingdom
535330,581207,21688,mixed up,-337,2011-12-07 18:34:00,0.00,NaN,United Kingdom


In [6]:
# Investigate extreme positive quantities

orders.nlargest(20, "Quantity")[
    ["InvoiceNo", "StockCode", "Description", "Quantity",
     "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
74614,542504,37413,NaN,5568,2011-01-28 12:03:00,0.00,NaN,United Kingdom
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901.0,United Kingdom
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135.0,United Kingdom
220843,556231,85123A,?,4000,2011-06-09 15:04:00,0.00,NaN,United Kingdom
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087.0,United Kingdom
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,2011-07-19 17:04:00,0.06,14609.0,United Kingdom
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom


## Data Cleaning Strategy

Based on the data-quality investigation, the following rules were established for customer-level analysis:

1. Exact duplicate rows will be removed because they do not provide additional information.

2. Cancellation invoices, identified by InvoiceNo beginning with "C", will be excluded from purchase-based customer metrics.

3. Negative-quantity transactions that are not cancellation invoices will be treated as operational adjustments rather than customer purchases. These include records associated with damaged, discarded, lost, or incorrectly recorded items.

4. Transactions with zero or negative UnitPrice will be excluded from customer revenue calculations because they do not represent standard positive-value purchases.

5. Transactions without CustomerID will be excluded from customer-level analysis because they cannot be reliably attributed to an individual customer. They may still be retained in the raw transaction data for other forms of analysis.

6. Extreme positive quantities will not be removed solely because they are statistical outliers. Where the transaction has a positive price and identifiable customer information, it may represent a legitimate bulk purchase.

7. Cancellation and adjustment records will be retained separately rather than permanently deleted, allowing them to be analyzed independently if required.

A Revenue field will be calculated as:

**Revenue = Quantity × UnitPrice**

In [7]:
# Remove exact duplicates
orders_clean_base = orders.drop_duplicates().copy()

# Identify cancellations
is_cancellation = (
    orders_clean_base["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

# Keep cancellations separately
cancellations = orders_clean_base[is_cancellation].copy()

# Identify non-cancellation operational adjustments
is_operational_adjustment = (
    (~is_cancellation) &
    (orders_clean_base["Quantity"] < 0)
)

adjustments = orders_clean_base[is_operational_adjustment].copy()

# Create customer purchase dataset
transactions_clean = orders_clean_base[
    (~is_cancellation) &
    (orders_clean_base["Quantity"] > 0) &
    (orders_clean_base["UnitPrice"] > 0) &
    (orders_clean_base["CustomerID"].notna())
].copy()

# Calculate revenue
transactions_clean["Revenue"] = (
    transactions_clean["Quantity"] *
    transactions_clean["UnitPrice"]
)

# Reset indexes
transactions_clean.reset_index(drop=True, inplace=True)
cancellations.reset_index(drop=True, inplace=True)
adjustments.reset_index(drop=True, inplace=True)

print("Original rows:", len(orders))
print("After removing duplicates:", len(orders_clean_base))
print("Cancellation rows:", len(cancellations))
print("Operational adjustment rows:", len(adjustments))
print("Clean customer transactions:", len(transactions_clean))

Original rows: 541909
After removing duplicates: 536641
Cancellation rows: 9251
Operational adjustment rows: 1336
Clean customer transactions: 392692


In [8]:
print("Missing CustomerID:",
      transactions_clean["CustomerID"].isna().sum())

print("Negative quantities:",
      (transactions_clean["Quantity"] < 0).sum())

print("Zero/negative prices:",
      (transactions_clean["UnitPrice"] <= 0).sum())

print("Duplicate rows:",
      transactions_clean.duplicated().sum())

print("Unique customers:",
      transactions_clean["CustomerID"].nunique())

print("Unique invoices:",
      transactions_clean["InvoiceNo"].nunique())

print("Total Revenue: £{:,.2f}".format(
    transactions_clean["Revenue"].sum()
))

Missing CustomerID: 0
Negative quantities: 0
Zero/negative prices: 0
Duplicate rows: 0
Unique customers: 4338
Unique invoices: 18532
Total Revenue: £8,887,208.89


## Customer Revenue Concentration

The first customer-level analysis examines how revenue is distributed across the customer base.

The objective is to determine whether revenue is broadly distributed or concentrated among a relatively small group of customers.

Understanding revenue concentration helps identify:
- High-value customer groups
- Potential customer dependency risk
- Customers that may warrant stronger retention efforts

In [9]:
# Calculate total revenue generated by each customer

customer_revenue = (
    transactions_clean
    .groupby("CustomerID")
    .agg(
        Revenue=("Revenue", "sum")
    )
    .reset_index()
    .sort_values("Revenue", ascending=False)
)

customer_revenue.head(10)

,CustomerID,Revenue
1689,14646.0,280206.02
4201,18102.0,259657.30
3728,17450.0,194390.79
3008,16446.0,168472.50
1879,14911.0,143711.17
55,12415.0,124914.53
1333,14156.0,117210.08
3771,17511.0,91062.38
2702,16029.0,80850.84
0,12346.0,77183.60


### Revenue Concentration

To quantify customer revenue concentration, customers will be ranked by total revenue and their cumulative contribution to overall revenue will be calculated.

This helps determine how dependent the business is on its highest-value customers and whether retention efforts should prioritize a relatively small group of customers.

In [10]:
# Calculate cumulative revenue contribution by customer

customer_revenue["Cumulative Revenue"] = customer_revenue["Revenue"].cumsum()

total_revenue = customer_revenue["Revenue"].sum()

customer_revenue["Cumulative Revenue %"] = (
    customer_revenue["Cumulative Revenue"] / total_revenue * 100
)

customer_revenue.head(10)

,CustomerID,Revenue,Cumulative Revenue,Cumulative Revenue %
1689,14646.0,280206.02,280206.02,3.152914
4201,18102.0,259657.30,539863.32,6.074610
3728,17450.0,194390.79,734254.11,8.261920
3008,16446.0,168472.50,902726.61,10.157594
1879,14911.0,143711.17,1046437.78,11.774650
55,12415.0,124914.53,1171352.31,13.180205
1333,14156.0,117210.08,1288562.39,14.499067
3771,17511.0,91062.38,1379624.77,15.523713
2702,16029.0,80850.84,1460475.61,16.433457
0,12346.0,77183.60,1537659.21,17.301936


### Revenue Contribution by Customer Group

To make the concentration analysis actionable, customers will be grouped into the top 1%, 5%, 10%, and 20% based on revenue.

This will show how much of the company's revenue is generated by its highest-value customer groups.

In [11]:
# Calculate revenue contribution of the top customer groups

customer_count = len(customer_revenue)

top_1_count = max(1, int(customer_count * 0.01))
top_5_count = max(1, int(customer_count * 0.05))
top_10_count = max(1, int(customer_count * 0.10))
top_20_count = max(1, int(customer_count * 0.20))

revenue_concentration = pd.DataFrame({
    "Customer Group": ["Top 1%", "Top 5%", "Top 10%", "Top 20%"],
    "Customers": [top_1_count, top_5_count, top_10_count, top_20_count],
    "Revenue": [
        customer_revenue.head(top_1_count)["Revenue"].sum(),
        customer_revenue.head(top_5_count)["Revenue"].sum(),
        customer_revenue.head(top_10_count)["Revenue"].sum(),
        customer_revenue.head(top_20_count)["Revenue"].sum()
    ]
})

revenue_concentration["Revenue Contribution %"] = (
    revenue_concentration["Revenue"] / total_revenue * 100
)

revenue_concentration

,Customer Group,Customers,Revenue,Revenue Contribution %
0,Top 1%,43,2829679.520,31.839912
1,Top 5%,216,4478213.760,50.389428
2,Top 10%,433,5457733.650,61.411110
3,Top 20%,867,6635245.311,74.660621


### Key Finding: Revenue Concentration

Customer revenue is highly concentrated within a relatively small portion of the customer base.

The top 1% of customers generate approximately 31.8% of total revenue, while the top 5% generate approximately 50.4%. The top 20% account for approximately 74.7% of revenue.

This concentration indicates that a relatively small group of customers has a significant impact on overall revenue, making customer retention and relationship management particularly important for high-value customers.

However, revenue alone does not indicate customer loyalty. The next analysis examines whether customers are making repeat purchases and how frequently they purchase.

## One-Time vs Repeat Customers

Revenue concentration shows that a relatively small group of customers contributes a large share of revenue.

The next step is to understand customer purchasing behavior by distinguishing between one-time and repeat customers.

This analysis will answer:

- How many customers made only one purchase?
- How many returned to make additional purchases?
- What share of the customer base is retained through repeat purchasing?

Repeat purchasing provides an initial view of customer retention before we move into more detailed measures of purchase frequency and customer lifecycle.

In [12]:
# Count the number of unique invoices for each customer

customer_frequency = (
    transactions_clean
    .groupby("CustomerID")
    .agg(
        Purchase_Count=("InvoiceNo", "nunique")
    )
    .reset_index()
)

customer_frequency["Customer_Type"] = np.where(
    customer_frequency["Purchase_Count"] == 1,
    "One-Time",
    "Repeat"
)

customer_frequency.head(10)

,CustomerID,Purchase_Count,Customer_Type
0,12346.0,1,One-Time
1,12347.0,7,Repeat
2,12348.0,4,Repeat
3,12349.0,1,One-Time
4,12350.0,1,One-Time
5,12352.0,8,Repeat
6,12353.0,1,One-Time
7,12354.0,1,One-Time
8,12355.0,1,One-Time
9,12356.0,3,Repeat


### One-Time vs Repeat Customer Distribution

Customers are classified based on the number of unique invoices associated with their purchases.

- **One-Time Customer:** 1 unique invoice
- **Repeat Customer:** More than 1 unique invoice

This analysis helps assess how much of the customer base demonstrates repeat purchasing behavior and provides an initial indication of customer retention.

In [13]:
# Summarize one-time vs repeat customers

customer_type_summary = (
    customer_frequency
    .groupby("Customer_Type")
    .agg(
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

customer_type_summary["Customer Share %"] = (
    customer_type_summary["Customers"]
    / customer_type_summary["Customers"].sum()
    * 100
)

customer_type_summary

,Customer_Type,Customers,Customer Share %
0,One-Time,1493,34.416782
1,Repeat,2845,65.583218


### Key Finding: Repeat Purchasing

Of the 4,338 identifiable customers in the cleaned purchase dataset, 2,845 customers (65.6%) made purchases across more than one invoice, while 1,493 customers (34.4%) made only one purchase.

This indicates that repeat purchasing is present across a substantial portion of the customer base. However, invoice count alone does not capture the intensity or consistency of customer engagement, so purchase frequency and customer value will be examined next.

In [14]:
# Calculate the average number of purchases per customer

purchase_frequency_summary = customer_frequency["Purchase_Count"].describe()

purchase_frequency_summary

count    4338.000000
mean        4.272015
std         7.697998
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       209.000000
Name: Purchase_Count, dtype: float64

### Purchase Frequency Distribution

Average purchase frequency can be influenced by a small number of highly active customers. Therefore, the distribution of purchase frequency is examined to understand how customer behavior varies across the customer base.

The analysis focuses on the median and distribution of invoices per customer rather than relying only on the average.

In [15]:
# Group customers by purchase frequency

frequency_distribution = (
    customer_frequency
    .groupby("Purchase_Count")
    .agg(
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

frequency_distribution["Customer Share %"] = (
    frequency_distribution["Customers"]
    / frequency_distribution["Customers"].sum()
    * 100
)

frequency_distribution.head(15)

,Purchase_Count,Customers,Customer Share %
0,1,1493,34.416782
1,2,835,19.248502
2,3,508,11.710466
3,4,388,8.944214
4,5,242,5.578608
5,6,172,3.964961
6,7,143,3.296450
7,8,98,2.259106
8,9,68,1.567543
9,10,54,1.244813


### Customer Frequency Segments

To make the purchase-frequency distribution more actionable, customers are grouped into four behavioral segments:

- **One-Time:** 1 purchase
- **Occasional:** 2–3 purchases
- **Regular:** 4–10 purchases
- **Frequent:** More than 10 purchases

These segments provide a clearer view of how customer engagement is distributed across the customer base.

In [16]:
# Create customer frequency segments

customer_frequency["Frequency_Segment"] = pd.cut(
    customer_frequency["Purchase_Count"],
    bins=[0, 1, 3, 10, float("inf")],
    labels=["One-Time", "Occasional", "Regular", "Frequent"]
)

frequency_segments = (
    customer_frequency
    .groupby("Frequency_Segment", observed=False)
    .agg(
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

frequency_segments["Customer Share %"] = (
    frequency_segments["Customers"]
    / frequency_segments["Customers"].sum()
    * 100
)

frequency_segments

,Frequency_Segment,Customers,Customer Share %
0,One-Time,1493,34.416782
1,Occasional,1343,30.958967
2,Regular,1165,26.855694
3,Frequent,337,7.768557


## Customer Value Distribution

Purchase frequency shows how often customers transact, but frequency alone does not indicate how valuable those customers are.

The next analysis combines customer revenue with purchasing behavior to understand the distribution of customer value.

This helps identify whether frequent customers are also generating significant revenue and whether a small group of customers contributes disproportionately to business value.

In [17]:
# Combine customer revenue and purchase frequency

customer_value = customer_revenue.merge(
    customer_frequency,
    on="CustomerID",
    how="inner"
)

customer_value.head(10)

,CustomerID,Revenue,Cumulative Revenue,Cumulative Revenue %,Purchase_Count,Customer_Type,Frequency_Segment
0,14646.0,280206.02,280206.02,3.152914,73,Repeat,Frequent
1,18102.0,259657.30,539863.32,6.074610,60,Repeat,Frequent
2,17450.0,194390.79,734254.11,8.261920,46,Repeat,Frequent
3,16446.0,168472.50,902726.61,10.157594,2,Repeat,Occasional
4,14911.0,143711.17,1046437.78,11.774650,201,Repeat,Frequent
5,12415.0,124914.53,1171352.31,13.180205,21,Repeat,Frequent
6,14156.0,117210.08,1288562.39,14.499067,55,Repeat,Frequent
7,17511.0,91062.38,1379624.77,15.523713,31,Repeat,Frequent
8,16029.0,80850.84,1460475.61,16.433457,63,Repeat,Frequent
9,12346.0,77183.60,1537659.21,17.301936,1,One-Time,One-Time


### Revenue by Customer Frequency Segment

To understand the relationship between purchasing frequency and customer value, total revenue will be compared across the four customer frequency segments.

This will show whether the business's revenue is primarily generated by frequent customers or whether lower-frequency customers also contribute substantially to revenue.

In [18]:
# Compare revenue across customer frequency segments

frequency_value = (
    customer_value
    .groupby("Frequency_Segment", observed=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

frequency_value["Customer Share %"] = (
    frequency_value["Customers"]
    / frequency_value["Customers"].sum()
    * 100
)

frequency_value["Revenue Share %"] = (
    frequency_value["Revenue"]
    / frequency_value["Revenue"].sum()
    * 100
)

frequency_value

,Frequency_Segment,Customers,Revenue,Customer Share %,Revenue Share %
0,One-Time,1493,613989.561,34.416782,6.908688
1,Occasional,1343,1386584.392,30.958967,15.602023
2,Regular,1165,2502668.931,26.855694,28.160348
3,Frequent,337,4383966.010,7.768557,49.328941


### Key Finding: Customer Frequency and Revenue

Customer frequency is strongly associated with revenue contribution.

Frequent customers represent only 7.8% of the customer base but generate 49.3% of total revenue. In contrast, one-time customers represent 34.4% of customers but contribute only 6.9% of revenue.

This indicates that a relatively small group of highly active customers is responsible for a substantial share of business revenue.

The next step is to examine customer value at an individual level and identify whether high-frequency customers consistently generate higher revenue per customer.

### Individual Customer Revenue Distribution

Customer revenue is highly skewed, with a relatively small number of customers generating substantially more revenue than the typical customer.

The following distribution examines customer-level revenue using summary statistics and percentiles to distinguish typical customer value from the extreme high-value end of the customer base.

In [19]:
# Examine the distribution of individual customer revenue

customer_revenue["Revenue"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count      4338.000000
mean       2048.688081
std        8985.230220
min           3.750000
25%         306.482500
50%         668.570000
75%        1660.597500
90%        3640.841000
95%        5789.999500
99%       19780.487800
max      280206.020000
Name: Revenue, dtype: float64

## First-to-Second Purchase Timing

Repeat purchasing is not only about whether customers return, but also how quickly they make their next purchase.

This analysis measures the time between a customer's first and second purchase to identify the typical repeat-purchase window.

In [28]:
# Create customer-level purchase dates
customer_purchases = (
    transactions_clean
    .groupby(["CustomerID", "InvoiceNo"], as_index=False)
    .agg(
        Purchase_Date=("InvoiceDate", "min")
    )
)

# Rank purchases using the same logic as the SQL analysis:
# purchase date determines the sequence
customer_purchases = customer_purchases.sort_values(
    ["CustomerID", "Purchase_Date", "InvoiceNo"]

)

customer_purchases["Purchase_Number"] = (
    customer_purchases
    .groupby("CustomerID")
    .cumcount() + 1
)

# First purchase
first_purchase = (
    customer_purchases[
        customer_purchases["Purchase_Number"] == 1
    ][["CustomerID", "Purchase_Date"]]
    .rename(
        columns={"Purchase_Date": "First_Purchase"}
    )
)

# Second purchase
second_purchase = (
    customer_purchases[
        customer_purchases["Purchase_Number"] == 2
    ][["CustomerID", "Purchase_Date"]]
    .rename(
        columns={"Purchase_Date": "Second_Purchase"}
    )
)

# Combine first and second purchases
second_purchase_timing = first_purchase.merge(
    second_purchase,
    on="CustomerID",
    how="inner"
)

# Exact elapsed time in days
second_purchase_timing["Days_to_Second_Purchase"] = (
    second_purchase_timing["Second_Purchase"]
    - second_purchase_timing["First_Purchase"]
).dt.total_seconds() / 86400

print(
    f"Repeat customers analyzed: "
    f"{len(second_purchase_timing):,}"
)

print(
    f"Average days to second purchase: "
    f"{second_purchase_timing['Days_to_Second_Purchase'].mean():.1f}"
)

print(
    f"Median days to second purchase: "
    f"{second_purchase_timing['Days_to_Second_Purchase'].median():.1f}"
)

second_purchase_timing.head()

Repeat customers analyzed: 2,845
Average days to second purchase: 76.7
Median days to second purchase: 50.1


,CustomerID,First_Purchase,Second_Purchase,Days_to_Second_Purchase
0,12347.0,2010-12-07 14:57:00,2011-01-26 14:30:00,49.981250
1,12348.0,2010-12-16 19:09:00,2011-01-25 10:42:00,39.647917
2,12352.0,2011-02-16 12:33:00,2011-03-01 14:57:00,13.100000
3,12356.0,2011-01-18 09:50:00,2011-04-08 12:33:00,80.113194
4,12358.0,2011-07-12 10:04:00,2011-12-08 10:26:00,149.015278


In [21]:
# Purchase timing distribution
# Whole calendar days are used for the timing bands,
# matching the SQL DATEDIFF logic.

second_purchase_timing["Timing_Days"] = (
    second_purchase_timing["Second_Purchase"]
    - second_purchase_timing["First_Purchase"]
).dt.days

second_purchase_timing["Timing_Band"] = pd.cut(
    second_purchase_timing["Timing_Days"],
    bins=[-1, 7, 30, 60, 90, 180, float("inf")],
    labels=[
        "0-7 days",
        "8-30 days",
        "31-60 days",
        "61-90 days",
        "91-180 days",
        "181+ days"
    ]
)

timing_distribution = (
    second_purchase_timing
    .groupby("Timing_Band", observed=False)
    .agg(
        Customers=("CustomerID", "count")
    )
    .reset_index()
)

timing_distribution["Share"] = (
    timing_distribution["Customers"]
    / timing_distribution["Customers"].sum()
)

timing_distribution["Share"] = (
    timing_distribution["Share"] * 100
)

timing_distribution

,Timing_Band,Customers,Share
0,0-7 days,441,15.500879
1,8-30 days,537,18.875220
2,31-60 days,606,21.300527
3,61-90 days,354,12.442882
4,91-180 days,590,20.738137
5,181+ days,317,11.142355


## Customer RFM Segmentation

RFM analysis combines **recency, purchase frequency, and monetary value** to identify differences in customer behavior and value.

- **Recency:** How recently the customer purchased
- **Frequency:** Number of distinct purchases
- **Monetary:** Total revenue generated

Customers are scored into quintiles for each dimension, with higher scores representing more favorable customer behavior.

In [22]:
# RFM customer segmentation

snapshot_date = transactions_clean["InvoiceDate"].max()

rfm = (
    transactions_clean
    .groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

# Create quintile scores
# Recency is reversed because fewer days since purchase is better.
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

# Business-oriented customer segments
def assign_rfm_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 4:
        return "Loyal Customers"
    elif r >= 4 and m >= 4:
        return "High Value - Recent"
    elif r <= 2 and m >= 4:
        return "High Value - At Risk"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r >= 4:
        return "Recent Customers"
    else:
        return "Developing Customers"

rfm["RFM_Segment"] = rfm.apply(assign_rfm_segment, axis=1)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Segment
0,12346.0,325,1,77183.60,1,1,5,115,High Value - At Risk
1,12347.0,1,7,4310.00,5,5,5,555,Champions
2,12348.0,74,4,1797.24,2,4,4,244,High Value - At Risk
3,12349.0,18,1,1757.55,4,1,4,414,High Value - Recent
4,12350.0,309,1,334.40,1,1,2,112,Developing Customers


In [23]:
# RFM segment summary

rfm_summary = (
    rfm
    .groupby("RFM_Segment")
    .agg(
        Customers=("CustomerID", "nunique"),
        Revenue=("Monetary", "sum"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
    .reset_index()
)

rfm_summary["Revenue_Share"] = (
    rfm_summary["Revenue"] / rfm_summary["Revenue"].sum() * 100
)

rfm_summary.sort_values("Revenue", ascending=False)

,RFM_Segment,Customers,Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary,Revenue_Share
1,Champions,942,5737952.120,11.503185,11.197452,6091.244289,64.564164
5,Loyal Customers,508,927623.751,36.206693,5.124016,1826.031006,10.437740
3,High Value - At Risk,346,851944.151,140.976879,3.765896,2462.266332,9.586183
2,Developing Customers,1533,626265.471,159.692759,1.252446,408.522812,7.046818
4,High Value - Recent,114,351913.470,15.052632,2.412281,3086.960263,3.959775
0,At Risk,395,196802.731,163.230380,2.683544,498.234762,2.214449
6,Recent Customers,500,194707.200,16.154000,1.646000,389.414400,2.190870


In [24]:
# RFM segment summary for business prioritization

rfm_priority = (
    rfm_summary
    .copy()
    .sort_values("Revenue", ascending=False)
)

rfm_priority["Customer_Share"] = (
    rfm_priority["Customers"]
    / rfm_priority["Customers"].sum()
    * 100
)

rfm_priority[
    [
        "RFM_Segment",
        "Customers",
        "Customer_Share",
        "Revenue",
        "Revenue_Share",
        "Avg_Recency",
        "Avg_Frequency",
        "Avg_Monetary"
    ]
]

,RFM_Segment,Customers,Customer_Share,Revenue,Revenue_Share,Avg_Recency,Avg_Frequency,Avg_Monetary
1,Champions,942,21.715076,5737952.120,64.564164,11.503185,11.197452,6091.244289
5,Loyal Customers,508,11.710466,927623.751,10.437740,36.206693,5.124016,1826.031006
3,High Value - At Risk,346,7.976026,851944.151,9.586183,140.976879,3.765896,2462.266332
2,Developing Customers,1533,35.338866,626265.471,7.046818,159.692759,1.252446,408.522812
4,High Value - Recent,114,2.627939,351913.470,3.959775,15.052632,2.412281,3086.960263
0,At Risk,395,9.105579,196802.731,2.214449,163.230380,2.683544,498.234762
6,Recent Customers,500,11.526049,194707.200,2.190870,16.154000,1.646000,389.414400


### RFM Business Finding

Champions represent 21.7% of customers but generate 64.6% of customer revenue, highlighting a strong concentration of value among a relatively small customer group.

High Value – At Risk customers account for 8.0% of customers and 9.6% of historical revenue. Their combination of high monetary value and weaker recency makes them a relevant retention priority.

In [25]:
# High-value customers at risk of inactivity

high_value_at_risk = (
    rfm[
        (rfm["RFM_Segment"] == "High Value - At Risk")
    ]
    .sort_values("Monetary", ascending=False)
    .copy()
)

print(f"High Value - At Risk customers: {len(high_value_at_risk):,}")
print(
    f"Historical revenue: £{high_value_at_risk['Monetary'].sum():,.2f}"
)
print(
    f"Average days since purchase: "
    f"{high_value_at_risk['Recency'].mean():.1f}"
)

high_value_at_risk.head(10)

High Value - At Risk customers: 346
Historical revenue: £851,944.15
Average days since purchase: 141.0


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Segment
0,12346.0,325,1,77183.60,1,1,5,115,High Value - At Risk
2502,15749.0,234,3,44534.30,1,3,5,135,High Value - At Risk
2011,15098.0,181,3,39916.50,1,3,5,135,High Value - At Risk
50,12409.0,78,3,11072.67,2,3,5,235,High Value - At Risk
2814,16180.0,99,8,10254.18,2,5,5,255,High Value - At Risk
196,12590.0,210,2,9864.26,1,2,5,125,High Value - At Risk
566,13093.0,275,8,7832.47,1,5,5,155,High Value - At Risk
73,12435.0,79,2,7829.89,2,2,5,225,High Value - At Risk
485,12980.0,157,9,7374.90,2,5,5,255,High Value - At Risk
3222,16745.0,86,17,7180.70,2,5,5,255,High Value - At Risk


In [26]:
# Inactivity profile of high-value at-risk customers

at_risk_inactivity = (
    high_value_at_risk
    .assign(
        Inactivity_Band=pd.cut(
            high_value_at_risk["Recency"],
            bins=[-1, 30, 60, 90, 180, float("inf")],
            labels=[
                "0-30 days",
                "31-60 days",
                "61-90 days",
                "91-180 days",
                "181+ days"
            ]
        )
    )
    .groupby("Inactivity_Band", observed=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        Historical_Revenue=("Monetary", "sum")
    )
    .reset_index()
)

at_risk_inactivity["Revenue_Share"] = (
    at_risk_inactivity["Historical_Revenue"]
    / at_risk_inactivity["Historical_Revenue"].sum()
    * 100
)

at_risk_inactivity

,Inactivity_Band,Customers,Historical_Revenue,Revenue_Share
0,0-30 days,0,0.000,0.000000
1,31-60 days,0,0.000,0.000000
2,61-90 days,103,232829.030,27.329142
3,91-180 days,158,304474.480,35.738784
4,181+ days,85,314640.641,36.932074


### At-Risk Customer Finding

The 346 High Value – At Risk customers account for £851.9K in historical revenue. All customers in this segment have been inactive for at least 61 days, with 72.7% of the segment's historical revenue coming from customers inactive for more than 90 days.

These customers represent a retention priority based on their historical value and current inactivity. The analysis does not assume that inactivity means confirmed churn.

## Cohort Retention Analysis

Cohort analysis tracks customer activity after the month of their first purchase. It shows how consistently customers from different acquisition cohorts remain active over time.

In [29]:
# Monthly cohort retention

cohort_data = transactions_clean[
    ["CustomerID", "InvoiceDate"]
].copy()

cohort_data["Purchase_Month"] = (
    cohort_data["InvoiceDate"]
    .dt.to_period("M")
)

cohort_data["Cohort_Month"] = (
    cohort_data
    .groupby("CustomerID")["Purchase_Month"]
    .transform("min")
)

cohort_data["Cohort_Index"] = (
    (cohort_data["Purchase_Month"].dt.year -
     cohort_data["Cohort_Month"].dt.year) * 12
    +
    (cohort_data["Purchase_Month"].dt.month -
     cohort_data["Cohort_Month"].dt.month)
    + 
)

cohort_counts = (
    cohort_data
    .groupby(["Cohort_Month", "Cohort_Index"])["CustomerID"]
    .nunique()
    .reset_index(name="Active_Customers")
)

cohort_sizes = (
    cohort_counts
    .groupby("Cohort_Month")["Active_Customers"]
    .first()
    .rename("Cohort_Size")
)

cohort_counts["Retention_Rate"] = (
    cohort_counts["Active_Customers"]
    / cohort_counts["Cohort_Month"].map(cohort_sizes)
    * 100
)

cohort_retention = (
    cohort_counts
    .pivot(
        index="Cohort_Month",
        columns="Cohort_Index",
        values="Retention_Rate"
    )
)

cohort_retention

SyntaxError: invalid syntax (3232506302.py, line 25)

### Cohort Retention Finding

Cohort activity retention varies substantially across acquisition months, with later-period activity fluctuating rather than declining in a strictly linear pattern. The cohort view highlights differences in repeat engagement over time and provides a basis for comparing customer retention across acquisition cohorts.

This analysis measures customer activity in each period after acquisition; it does not imply continuous month-by-month retention.



## Key Findings

- **Customer value is highly concentrated:** Champions represent 21.7% of customers but generate 64.6% of customer revenue.
- **Repeat purchasing is widespread but not universal:** 65.6% of customers made more than one purchase, while 34.4% purchased only once.
- **Frequent customers contribute disproportionately to revenue:** 7.8% of customers generated 49.3% of total revenue.
- **High-value inactivity represents a retention priority:** 346 High Value – At Risk customers account for £851.9K in historical revenue, with 72.7% of that revenue coming from customers inactive for more than 90 days.
- **The second purchase takes time:** Repeat customers took an average of 76.7 days to make their second purchase.
- **Customer engagement varies across cohorts:** Cohort activity retention differs across acquisition months, indicating variation in repeat engagement over time.


## Business Recommendations

1. **Protect high-value customers** by prioritizing Champions and other high-value customers in retention initiatives.
2. **Prioritize reactivation of high-value inactive customers**, particularly those with more than 90 days since their last purchase.
3. **Improve the path to the second purchase** by identifying opportunities to reduce the 76.7-day average gap between first and second purchases.
4. **Use cohort activity patterns to evaluate retention performance** across different acquisition periods.
5. **Focus retention efforts on value, not customer count alone**, given the strong concentration of revenue among higher-value customer groups.